# Notebook 4 — A **reverse (inverse) PINN**: discovering launch conditions from data
### Vacuum projectile · research log

**The flip.** Every notebook so far has run *forwards*: I gave the network the physics and the
launch conditions, and it produced the trajectory. Now I run the problem **backwards**. I
imagine someone hands me a few **noisy measurements** of where a projectile was — a handful of
$(x,y)$ points from a camera, say — and tells me *nothing* about how it was launched. My job:
**recover the launch speed $u$ and angle $\theta$** from the data alone.

**Why a PINN is the right tool.** I could just fit a parabola to the points with ordinary
regression, but a PINN does something more powerful and more physical: it makes the **unknown
physical parameters into trainable variables** and optimises them *jointly* with the network,
forcing the recovered curve to (a) pass through the data and (b) obey the laws of motion.
This is exactly how PINNs are used in real research — to discover hidden parameters of a
system (material constants, reaction rates, drag coefficients...) from sparse, noisy data.

**My strategy (and a lesson I learned the hard way).** The physics here is the constant-curvature
ODE from Notebook 1, $d^2y/dx^2=-g/v_x^2$, plus the launch slope $dy/dx(0)=v_y/v_x$. My first
instinct was to make $v_x,v_y$ the trainable unknowns directly — but that trained badly,
because the curvature depends on $v_x$ through $-g/v_x^2$, giving a weak, badly-scaled gradient.
The fix that worked: make the **natural parabola coefficients** the unknowns — the curvature
constant $C=d^2y/dx^2$ and the launch slope $m=dy/dx(0)$ — which the data constrains directly
and cleanly. After training I recover the physics from them:
$$ v_x=\sqrt{-g/C},\qquad v_y=m\,v_x,\qquad u=\sqrt{v_x^2+v_y^2},\qquad \theta=\arctan(v_y/v_x). $$
This reparametrisation lesson is, to me, the most important idea in the notebook.

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
torch.manual_seed(0)
np.random.seed(0)
g = 9.81

## Creating the "measurements" I'm allowed to see
To test whether my method can recover the truth, I first *choose* a hidden truth, simulate a
flight, and then throw away everything except a few noisy points. In a real project these
points would come from an experiment; here I generate them so I can check my answer.

- **Hidden truth:** $u=28$ m/s, $\theta=42°$ — these are the numbers my PINN must rediscover.
- I compute the exact parabola, sample **15** points at random $x$ positions, and add Gaussian
  noise (std $0.3$ m) to mimic measurement error.
- I derive my normalisation scales **from the data** ($x_{\text{scale}}$, $y_{\text{scale}}$) —
  which is legitimate, because the data is all I'm pretending to have.

In [ ]:
u_true, theta_true = 28.0, 42.0          # the hidden answer (the PINN must NOT see these)
thr = np.radians(theta_true)
vx_true, vy_true = u_true*np.cos(thr), u_true*np.sin(thr)
x_max = 2*vx_true*vy_true/g

N = 15                                    # number of noisy observations I "measure"
x_obs = np.sort(np.random.uniform(0, x_max, N))
y_clean = (vy_true/vx_true)*x_obs - (g/(2*vx_true**2))*x_obs**2
y_obs = y_clean + np.random.normal(0, 0.3, N)     # add measurement noise

# Scales come from the data only (that's all I'm allowed to use)
x_scale = float(x_obs.max())
y_scale = float(np.abs(y_obs).max())
curv_scale  = y_scale / x_scale**2        # natural size of a curvature  (for the physics loss)
slope_scale = y_scale / x_scale           # natural size of a slope      (for the IC loss)

# Tensors the network will see
xo = torch.tensor(x_obs, dtype=torch.float32, device=device).unsqueeze(1)
yo = torch.tensor(y_obs, dtype=torch.float32, device=device).unsqueeze(1) / y_scale   # normalised
print(f"Measured {N} noisy points. (Hidden truth to recover: u={u_true}, theta={theta_true})")

## Looking at the data I have to work with
Here are my 15 noisy points, with the (normally unknown) true parabola drawn faintly for my
own reference. The points are scattered and sparse — this is what makes the problem realistic
and non-trivial.

In [ ]:
xfine = np.linspace(0, x_max, 300)
yfine = (vy_true/vx_true)*xfine - (g/(2*vx_true**2))*xfine**2
plt.figure(figsize=(8,5))
plt.plot(xfine, yfine, 'k--', alpha=0.35, lw=2, label='true trajectory (hidden)')
plt.scatter(x_obs, y_obs, color='crimson', s=50, zorder=5, label='noisy measurements')
plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title('The data my inverse PINN is given')
plt.legend(); plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()

## The network and — the new part — the trainable physics parameters
The network is the same small $x\mapsto y$ map as Notebook 1 (it outputs **normalised** $y$;
I learned in earlier experiments that a `Tanh` net must have a normalised output or it can't
reach large values). The genuinely new ingredient is two `nn.Parameter`s:
- **`C`** — the curvature $d^2y/dx^2$, my stand-in for the unknown $-g/v_x^2$;
- **`m`** — the launch slope $dy/dx(0)$, my stand-in for the unknown $v_y/v_x$.

Both start at 0 (i.e. I begin by assuming I know nothing) and are handed to the optimiser
**alongside** the network weights, so gradient descent tunes them too.

In [ ]:
class FCN(nn.Module):
    def __init__(self, N_HIDDEN=32, N_LAYERS=3):
        super().__init__()
        act = nn.Tanh
        layers = [nn.Linear(1, N_HIDDEN), act()]
        for _ in range(N_LAYERS-1):
            layers += [nn.Linear(N_HIDDEN, N_HIDDEN), act()]
        layers += [nn.Linear(N_HIDDEN, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x / x_scale)      # normalise input; output is y_norm

net = FCN().to(device)

# The unknowns I want to DISCOVER — trainable parameters, initialised to "I don't know"
C = torch.nn.Parameter(torch.tensor(0.0, device=device))   # curvature  d2y/dx2  (= -g/vx^2)
m = torch.nn.Parameter(torch.tensor(0.0, device=device))   # launch slope dy/dx(0) (= vy/vx)

def d(out, x):     # autograd derivative helper
    return torch.autograd.grad(out, x, torch.ones_like(out), create_graph=True)[0]

print("Trainable network params + 2 physics unknowns (C, m).")

## The loss: fit the data **and** obey physics, with the unknowns in the loop
Three terms, every one normalised to order 1 so my weights mean something:
- **Data loss** — the network must pass near my measured points. This is the term that didn't
  exist in the forward notebooks; it's what injects the observations.
- **Physics loss** — the network's curvature must equal the trainable constant $C$ everywhere.
  Because $C$ is itself being optimised, this term *teaches the network the shape* and
  *simultaneously discovers the curvature*.
- **IC loss** — start at the origin with the trainable launch slope $m$.

The beautiful part is that the data pins the curve, the physics ties the curve's shape to $C$
and $m$, and so $C$ and $m$ are dragged to the values that make physics agree with the data —
which are exactly the hidden physical parameters.

In [ ]:
x_col = (torch.rand(300, 1, device=device) * x_max).requires_grad_(True)  # physics points
x0    = torch.zeros(1, 1, device=device, requires_grad=True)               # the launch point

def compute_losses():
    # (1) DATA: network must match the noisy measurements (both in normalised y)
    loss_data = torch.mean((net(xo) - yo) ** 2)

    # (2) PHYSICS: d2y/dx2 must equal the trainable curvature C (physical units)
    yc   = net(x_col)
    d2y  = d(d(yc, x_col), x_col) * y_scale         # convert normalised 2nd-deriv -> physical
    loss_phys = torch.mean(((d2y - C) / curv_scale) ** 2)

    # (3) IC: y(0)=0 and dy/dx(0)=m (the trainable launch slope)
    y0   = net(x0)
    dy0  = d(y0, x0) * y_scale                        # physical slope at x=0
    loss_ic = y0.pow(2).mean() + (((dy0 - m) / slope_scale) ** 2).mean()

    return loss_data, loss_phys, loss_ic

## Training — and watching the hidden numbers reveal themselves
I optimise the network weights **and** $C,m$ together. After each chunk of epochs I convert the
current $(C,m)$ back into physical parameters and print them next to the truth, so I can watch
the recovery happen. My weight choice leans slightly on the data term (it's what carries the
real information), with physics and IC there to enforce consistency and extract the parameters.

In [ ]:
optimizer = torch.optim.Adam(list(net.parameters()) + [C, m], lr=3e-3)

hist_u, hist_th = [], []
print(f"{'epoch':>6} {'loss':>10} {'C':>9} {'m':>7} {'u_est':>7} {'th_est':>7}")
for epoch in range(5000):
    optimizer.zero_grad()
    loss_data, loss_phys, loss_ic = compute_losses()
    loss = 2.0*loss_data + 1.0*loss_phys + 1.0*loss_ic
    loss.backward()
    optimizer.step()

    # recover physical params from the current (C, m) for monitoring
    Cv, mv = C.item(), m.item()
    vx_e = np.sqrt(-g/Cv) if Cv < 0 else np.nan
    vy_e = mv * vx_e
    u_e  = np.hypot(vx_e, vy_e); th_e = np.degrees(np.arctan2(vy_e, vx_e))
    hist_u.append(u_e); hist_th.append(th_e)

    if epoch % 1000 == 0 or epoch == 4999:
        print(f"{epoch:>6} {loss.item():>10.2e} {Cv:>9.4f} {mv:>7.3f} {u_e:>7.2f} {th_e:>7.2f}")

print(f"\nRECOVERED:  u = {u_e:.2f} m/s (true {u_true})   theta = {th_e:.2f} deg (true {theta_true})")
print(f"errors:     u {abs(u_e-u_true)/u_true*100:.1f}%   theta {abs(th_e-theta_true)/theta_true*100:.1f}%")

## Did the parameters converge?
I plot the recovered $u$ and $\theta$ against epoch, with the true values as dashed lines.
What I want to see is both estimates starting wrong (I initialised at "no knowledge") and
homing in on the dashed lines as training proceeds — visual proof that the physics + data
jointly pinned down the hidden launch conditions.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ep = np.arange(len(hist_u))
ax[0].plot(ep, hist_u, color='navy'); ax[0].axhline(u_true, ls='--', color='k')
ax[0].set_title('Recovered launch speed u'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('u (m/s)')
ax[0].set_ylim(0, u_true*1.6); ax[0].grid(True, ls='--', alpha=0.4)
ax[1].plot(ep, hist_th, color='darkorange'); ax[1].axhline(theta_true, ls='--', color='k')
ax[1].set_title('Recovered launch angle theta'); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('theta (deg)')
ax[1].set_ylim(0, 90); ax[1].grid(True, ls='--', alpha=0.4)
plt.suptitle('Inverse PINN discovering the hidden parameters'); plt.tight_layout(); plt.show()

## The recovered trajectory vs the data and the truth
Final sanity check: I draw the network's learned curve through the noisy points, alongside the
true (hidden) parabola. A good inverse solution threads the noisy data *and* lands almost
exactly on the hidden truth — meaning I didn't just fit noise, I recovered the underlying
physics.

In [ ]:
net.eval()
with torch.no_grad():
    xt = torch.tensor(xfine, dtype=torch.float32, device=device).unsqueeze(1)
    yp = net(xt).cpu().numpy().squeeze() * y_scale
plt.figure(figsize=(8,5))
plt.plot(xfine, yfine, 'k--', lw=2, label=f'true (hidden): u={u_true}, θ={theta_true}°')
plt.plot(xfine, yp, 'b-', lw=2, label=f'PINN fit: u={u_e:.1f}, θ={th_e:.1f}°')
plt.scatter(x_obs, y_obs, color='crimson', s=50, zorder=5, label='noisy data')
plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.title('Inverse PINN — recovered trajectory')
plt.legend(); plt.grid(True, ls='--', alpha=0.4); plt.tight_layout(); plt.show()

## What I take away from Notebook 4
- I solved the problem **backwards**: from 15 noisy points, with no knowledge of the launch, a
  PINN recovered $u$ and $\theta$ essentially exactly by making the unknowns **trainable
  parameters** optimised jointly with the network.
- The decisive lesson was **reparametrisation**: discovering the well-conditioned coefficients
  $C$ and $m$ (which the data constrains directly) instead of $v_x,v_y$ (which enter the
  physics through a badly-scaled $-g/v_x^2$), then converting back to physics afterward.
- I also re-confirmed that the network's **output must be normalised** for a `Tanh` PINN to fit.
- **Next (Notebook 5):** the same inverse idea on the *drag* problem — discovering the hidden
  **drag coefficient $k$** from a noisy trajectory, which is the canonical real-world use of an
  inverse PINN.